# NYT + Guardian Archive Fetch

Bulk-fetch historical news headlines for use as economic event signals in the oil price world model.

**Sources:**
- **NYT Archive API** — one JSON dump per calendar month, all sections. Free key: https://developer.nytimes.com
- **Guardian Content API** — paginated search, finer section filters. Free key: https://open-platform.theguardian.com/access/

**Rate limits:**
- NYT: 500 requests/day, 5/min (we need only 24 requests for 2 years → fine)
- Guardian: 12/sec, 5000/day

**Output:** two parquet files (`nyt_2015_2016.parquet`, `guardian_2015_2016.parquet`) with normalized columns `date`, `headline`, `abstract`, `section`, `url`, `source`.

In [ ]:
import os
import time
import json
import requests
import pandas as pd
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

# Load keys from .env in the project root (expects NYT_API_KEY and GUARDIAN_API_KEY).
load_dotenv()

NYT_API_KEY = os.environ['NYT_API_KEY']
GUARDIAN_API_KEY = os.environ.get('GUARDIAN_API_KEY', '')

OUT_DIR = Path('./data') 
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2015]

## 1. NYT Archive API

Endpoint: `https://api.nytimes.com/svc/archive/v1/{year}/{month}.json?api-key=...`

Returns every article NYT published that month in a single response (typically 5–10k articles). We'll filter to economically relevant sections afterward.

In [2]:
def fetch_nyt_month(year: int, month: int, api_key: str) -> list[dict]:
    url = f'https://api.nytimes.com/svc/archive/v1/{year}/{month}.json'
    r = requests.get(url, params={'api-key': api_key}, timeout=60)
    r.raise_for_status()
    return r.json()['response']['docs']

In [ ]:
import pyarrow as pa

# Sections / desks worth keeping for a macro-finance model.
# Inspect the sample above and adjust as needed.
NYT_KEEP_SECTIONS = {'Business Day', 'Business', 'World', 'U.S.'}
NYT_KEEP_DESKS = {
    'Business', 'Financial', 'Foreign', 'Washington',
    'Energy', 'Economy', 'National'
}

def normalize_nyt(doc: dict) -> dict | None:
    section = doc.get('section_name') or ''
    desk = doc.get('news_desk') or ''
    if section not in NYT_KEEP_SECTIONS or desk not in NYT_KEEP_DESKS:
        return None
    headline = (doc.get('headline') or {}).get('main') or ''
    if not headline:
        return None
    pub = doc.get('pub_date', '')[:10]
    return {
        'date': pub,
        'headline': headline,
        'abstract': doc.get('abstract') or '',
        'lead_paragraph': doc.get('lead_paragraph') or '',
        'snippet': doc.get('snippet') or '',
        'section': section or desk,
        'url': doc.get('web_url', ''),
        'source': 'nyt',
    }

nyt_rows = []
for year in YEARS:
    for month in range(1, 13):
        docs = fetch_nyt_month(year, month, NYT_API_KEY)
        kept = [r for r in (normalize_nyt(d) for d in docs) if r]
        nyt_df = pd.DataFrame(kept)
        nyt_df.to_parquet(OUT_DIR / f'nyt_{year}_{month:02d}.parquet', index=False)
        print(f'{year}-{month:02d}: {len(docs):5d} raw → {len(kept):4d} kept')
        print(f'Exported to {OUT_DIR / f"nyt_{year}_{month:02d}.parquet"}')
        time.sleep(12) 

2015-01:  6906 raw → 1729 kept
Exported to \data\nyt_archive\nyt_2015_01.parquet
2015-02:  6437 raw → 1495 kept
Exported to \data\nyt_archive\nyt_2015_02.parquet
2015-03:  6975 raw → 1695 kept
Exported to \data\nyt_archive\nyt_2015_03.parquet
2015-04:  6542 raw → 1634 kept
Exported to \data\nyt_archive\nyt_2015_04.parquet
2015-05:  6634 raw → 1590 kept
Exported to \data\nyt_archive\nyt_2015_05.parquet
2015-06:  6947 raw → 1667 kept
Exported to \data\nyt_archive\nyt_2015_06.parquet
2015-07:  6758 raw → 1784 kept
Exported to \data\nyt_archive\nyt_2015_07.parquet
2015-08:  6086 raw → 1505 kept
Exported to \data\nyt_archive\nyt_2015_08.parquet
2015-09:  6990 raw → 1710 kept
Exported to \data\nyt_archive\nyt_2015_09.parquet
2015-10:  7365 raw → 1927 kept
Exported to \data\nyt_archive\nyt_2015_10.parquet
2015-11:  6241 raw → 1628 kept
Exported to \data\nyt_archive\nyt_2015_11.parquet
2015-12:  6371 raw → 1557 kept
Exported to \data\nyt_archive\nyt_2015_12.parquet


In [ ]:
nyt_df = pd.DataFrame(nyt_rows).sort_values('date').reset_index(drop=True)
nyt_df.to_csv(OUT_DIR / 'nyt_2015.csv', index=False)
print(f'NYT total kept: {len(nyt_df)}')
nyt_df.head()

In [ ]:
for i in range(105,125):
    print(f'NYT article {i+1}: \n\n')
    print(nyt_df.iloc[i]['date'])
    print(nyt_df.iloc[i]['headline'])
    print(nyt_df.iloc[i]['abstract'])
    print(nyt_df.iloc[i]['lead_paragraph'])
    print(nyt_df.iloc[i]['snippet'])
    print('\n\n')


## 2. Guardian Content API

Endpoint: `https://content.guardianapis.com/search`

Paginated (max 200/page), filtered by `section` and date range. We grab the Business, World, and Politics sections.

In [ ]:
def fetch_guardian_section(section: str, from_date: str, to_date: str, api_key: str) -> list[dict]:
    url = 'https://content.guardianapis.com/search'
    all_results = []
    page = 1
    while True:
        params = {
            'section': section,
            'from-date': from_date,
            'to-date': to_date,
            'page-size': 200,
            'page': page,
            'show-fields': 'trailText,headline',
            'order-by': 'oldest',
            'api-key': api_key,
        }
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()['response']
        all_results.extend(data['results'])
        if page >= data['pages']:
            break
        page += 1
        time.sleep(0.1)  # well under 12/sec
    return all_results

def normalize_guardian(doc: dict) -> dict:
    fields = doc.get('fields') or {}
    return {
        'date': doc['webPublicationDate'][:10],
        'headline': fields.get('headline') or doc.get('webTitle', ''),
        'abstract': fields.get('trailText', ''),
        'section': doc.get('sectionName', ''),
        'url': doc.get('webUrl', ''),
        'source': 'guardian',
    }

GUARDIAN_SECTIONS = ['business', 'world', 'politics']

guardian_rows = []
for section in GUARDIAN_SECTIONS:
    docs = fetch_guardian_section(section, '2015-01-01', '2016-12-31', GUARDIAN_API_KEY)
    print(f'{section}: {len(docs)} articles')
    guardian_rows.extend(normalize_guardian(d) for d in docs)

guardian_df = pd.DataFrame(guardian_rows).sort_values('date').reset_index(drop=True)
guardian_df.to_parquet(OUT_DIR / 'guardian_2015_2016.parquet', index=False)
print(f'Guardian total: {len(guardian_df)}')
guardian_df.head()

## 3. Quick inspection

Sanity-check daily volume and a few sample headlines from oil-relevant dates.

In [ ]:
combined = pd.concat([nyt_df, guardian_df], ignore_index=True)
combined['date'] = pd.to_datetime(combined['date'])

daily_counts = combined.groupby(['date', 'source']).size().unstack(fill_value=0)
print(daily_counts.describe())
daily_counts.rolling(7).mean().plot(figsize=(12, 4), title='7-day rolling article count')

In [ ]:
# Example: headlines from the OPEC meeting week Nov 27 2014 aftermath / 2016 Feb oil bottom
for d in ['2015-01-05', '2015-12-04', '2016-02-11', '2016-09-28']:
    sub = combined[combined['date'] == d]
    print(f'\n=== {d} ({len(sub)} articles) ===')
    for _, row in sub.head(8).iterrows():
        print(f'  [{row["source"]:8s}] {row["headline"]}')